## Contraindication IE + Normalization: VO

### 2026/04/23

### VO Statistics: 
1. Number of VO classes mapped to RxNorm : 2060
2. Number of SPL documents associated : 279

In [42]:
from owlready2 import * 
from owlready2 import IRIS
import pandas as pd
import requests

In [ ]:
onto = get_ontology("vo_source/VO_2026-03-06.owl").load()

In [55]:
res = []
for cls in onto.classes():
    if cls.VO_0003198: #RxNorm Annotation
        id = cls.name
        label = cls.label[0] if cls.label else None
        rx = cls.VO_0003198[0] if cls.VO_0003198 else None
        res.append((id, label, rx))


In [56]:
print(f"Number of classes with RxNorm annotation: {len(res)}")

Number of classes with RxNorm annotation: 2060


In [ ]:
# Extract and normalize contraindications of SPLs related to RxNorm annotations: 

In [57]:
df = pd.DataFrame(res, columns=["Class ID", "Class Label", "RxNorm Code"])
print(df.head())

     Class ID                                        Class Label RxNorm Code
0  VO_0003384                tetanus toxoid vaccine, inactivated      798306
1  VO_0003455             diphtheria toxoid vaccine, inactivated      798304
2  VO_0004208      BACILLUS ANTHRACIS STRAIN V770-NP1-R ANTIGENS     1368371
3  VO_0003150                Hepatitis B surface antigen vaccine      797752
4  VO_0003408  Haemophilus influenzae b (Ross strain) capsula...      798444


In [52]:
# API request for RxNorm
def get_rx_prop(rxnorm_code, property_name="allProperties"):
    base_url = f"https://rxnav.nlm.nih.gov"
    url = f"{base_url}/REST/rxcui/{rxnorm_code}/property.json"
    params = {"propName": property_name}
    response = requests.get(url, params=params)
    res = response.json()
    # resu = res.get("propConceptGroup", {}).get("propConcept", [])[0]
    result = [r.get("propValue") for r in res.get("propConceptGroup", {}).get("propConcept", [])]
    return result


In [58]:
df['spl'] = df['RxNorm Code'].apply(lambda x: get_rx_prop(x, "SPL_SET_ID"))

In [59]:
df

,Class ID,Class Label,RxNorm Code,spl
0,VO_0003384,"tetanus toxoid vaccine, inactivated",798306,"[06f34d0f-4e72-41d3-967f-8abf3f2005c1, 1554cd2..."
1,VO_0003455,"diphtheria toxoid vaccine, inactivated",798304,"[06f34d0f-4e72-41d3-967f-8abf3f2005c1, 174faf8..."
2,VO_0004208,BACILLUS ANTHRACIS STRAIN V770-NP1-R ANTIGENS,1368371,"[d4f29180-7c76-40ac-9341-cf62702c4090, e0b1180..."
3,VO_0003150,Hepatitis B surface antigen vaccine,797752,"[174faf80-eb4f-4f11-b875-d74bd26c2e65, 3ed8472..."
4,VO_0003408,Haemophilus influenzae b (Ross strain) capsula...,798444,"[174faf80-eb4f-4f11-b875-d74bd26c2e65, 3ed8472..."
...,...,...,...,...
2055,VO_0021174,"0.3 ML SARS-CoV-2 (COVID-19) vaccine, mRNA-BNT...",2664846,[]
2056,VO_0021176,"0.5 ML SARS-CoV-2 (COVID-19) vaccine, mRNA-127...",2664819,[]
2057,VO_0021177,"0.5 ML SARS-CoV-2 (COVID-19) vaccine, mRNA-127...",2664811,[]
2058,VO_0021178,"0.5 ML SARS-CoV-2 (COVID-19) vaccine, mRNA-127...",2664822,[]


In [69]:
reverse_lookup = df[df['spl'].map(len) > 0].explode('spl').set_index('spl')['RxNorm Code'].to_dict()

In [72]:
reverse_lookup_df = pd.DataFrame(list(reverse_lookup.items()), columns=['SPL_SET_ID', 'RxNorm Code'])
print(reverse_lookup_df.head())

                             SPL_SET_ID RxNorm Code
0  06f34d0f-4e72-41d3-967f-8abf3f2005c1     1300310
1  1554cd2d-e064-46f1-a635-2c80191a7b9e        7288
2  174faf80-eb4f-4f11-b875-d74bd26c2e65     2468239
3  1a87f53c-0c8a-4c66-9048-2a0cf60a6a5c     1190919
4  1d8930f5-434d-4bfa-befc-b54913c8dd36     1300205


In [73]:
print(f"Unique SPL_SET_IDs: {len(reverse_lookup_df['SPL_SET_ID'].unique())}")

Unique SPL_SET_IDs: 279


In [ ]:
# create reverse mapping from SPL_SET_ID to RxNorm Code using df 
spl_to_rx = {}
for _, row in df.iterrows():
    rx_code = row['RxNorm Code']
    spl_ids = row['spl']
    for spl_id in spl_ids:
        spl_to_rx[spl_id] = rx_code
print(spl_to_rx)